In [3]:
import pandas as pd
import numpy as np
import gc
import os
from google.colab import drive

# 1. MONTAJE DE DRIVE
print("📂 Montando Google Drive...")
drive.mount('/content/drive')

# --- CONFIGURACIÓN DE RUTAS ---
# Ajusta esta ruta si tu carpeta 'bt' está dentro de 'MyDrive'
FILE_PATH = '/content/drive/MyDrive/bt/ARC_FINAL_PRO_DATA.csv'
SAVE_PATH = '/content/drive/MyDrive/bt/ARC_AUDITORIA_RESULTADOS.csv'

# --- PARÁMETROS DE LA TESIS (v43.23) ---
SL_PTS = 30.0
TP_PTS = 60.0
LOOK_AHEAD = 400  # ticks hacia el futuro para validar el Spike
CHUNK_SIZE = 500000 # filas por bloque para no saturar la RAM

def ejecutar_auditoria():
    if not os.path.exists(FILE_PATH):
        print(f"❌ Error: No se encuentra el archivo en {FILE_PATH}")
        return

    print(f"🚀 Iniciando Auditoría Forense sobre {FILE_PATH}")

    # Columnas según la Firma Canónica de tu Logger
    cols = ['Time', 'Price', 'NodeM2', 'NodeH12', 'Upsilon', 'Energy', 'ConfUP']
    dtypes = {
        'Price': 'float32', 'NodeM2': 'float32', 'NodeH12': 'float32',
        'Upsilon': 'float32', 'Energy': 'float32', 'ConfUP': 'int8'
    }

    resultados = []

    # Procesamiento por trozos
    reader = pd.read_csv(FILE_PATH, sep=';', names=cols, header=0,
                         dtype=dtypes, chunksize=CHUNK_SIZE, engine='c')

    for i, chunk in enumerate(reader):
        # Filtro de Gatillo v43.23
        mask = (chunk['NodeM2'] >= 90) & (chunk['Energy'] >= 25000) & (chunk['Upsilon'] <= 0.65)
        indices = chunk.index[mask].tolist()

        for idx in indices:
            # Evitar desbordamiento al final del bloque
            if idx + LOOK_AHEAD >= chunk.index[-1]: continue

            entry_price = chunk.at[idx, 'Price']
            # Ventana de tiempo futura
            future = chunk.loc[idx+1 : idx+LOOK_AHEAD, 'Price']

            # Cálculo de Máximos y Mínimos (Drawdown y Runup)
            max_p = future.max()
            min_p = future.min()

            # ¿Qué tocó primero?
            tp_hit = future[future >= entry_price + TP_PTS].index
            sl_hit = future[future <= entry_price - SL_PTS].index

            idx_tp = tp_hit[0] if not tp_hit.empty else float('inf')
            idx_sl = sl_hit[0] if not sl_hit.empty else float('inf')

            status = "PERDIDA"
            if idx_tp < idx_sl: status = "GANADA"
            elif idx_tp == float('inf') and idx_sl == float('inf'): status = "EXPIRADA"

            resultados.append({
                'Time': chunk.at[idx, 'Time'],
                'Energy': chunk.at[idx, 'Energy'],
                'NodeH12': chunk.at[idx, 'NodeH12'],
                'Upsilon': chunk.at[idx, 'Upsilon'],
                'Resultado': status,
                'Max_Drawdown': round(entry_price - min_p, 2),
                'Max_Runup': round(max_p - entry_price, 2)
            })

        print(f"✅ Bloque {i+1} analizado... Señales acumuladas: {len(resultados)}")
        del chunk
        gc.collect()

    # 3. GENERACIÓN DE INFORME ESTRATÉGICO
    df_final = pd.DataFrame(resultados)

    if df_final.empty:
        print("❌ No se detectaron señales con los parámetros actuales.")
        return

    win_rate = (len(df_final[df_final['Resultado'] == 'GANADA']) / len(df_final)) * 100
    avg_dd = df_final['Max_Drawdown'].mean()
    avg_runup = df_final['Max_Runup'].mean()

    print("\n" + "═"*45)
    print("📊 REPORTE DE INTELIGENCIA ESTRATÉGICA ARC")
    print(f"🎯 Total de Gatillos: {len(df_final)}")
    print(f"🏆 Win Rate Potencial: {win_rate:.2f}%")
    print(f"📉 Drawdown Medio tras disparo: {avg_dd:.2f} pts")
    print(f"📈 Recorrido Medio (Runup): {avg_runup:.2f} pts")
    print("═"*45)

    # Guardar auditoría para análisis profundo
    df_final.to_csv(SAVE_PATH, index=False)
    print(f"💾 Resultados guardados en: {SAVE_PATH}")

ejecutar_auditoria()

📂 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando Auditoría Forense sobre /content/drive/MyDrive/bt/ARC_FINAL_PRO_DATA.csv
✅ Bloque 1 analizado... Señales acumuladas: 126727
✅ Bloque 2 analizado... Señales acumuladas: 255887
✅ Bloque 3 analizado... Señales acumuladas: 399110
✅ Bloque 4 analizado... Señales acumuladas: 543200
✅ Bloque 5 analizado... Señales acumuladas: 684463
✅ Bloque 6 analizado... Señales acumuladas: 830638
✅ Bloque 7 analizado... Señales acumuladas: 962439
✅ Bloque 8 analizado... Señales acumuladas: 1102032
✅ Bloque 9 analizado... Señales acumuladas: 1235442
✅ Bloque 10 analizado... Señales acumuladas: 1362356
✅ Bloque 11 analizado... Señales acumuladas: 1489509
✅ Bloque 12 analizado... Señales acumuladas: 1632175
✅ Bloque 13 analizado... Señales acumuladas: 1769189
✅ Bloque 14 analizado... Señales acumuladas: 1899192
✅ Bloque 15 analizado... Señales 